# Whisper dialect ASR — evaluation

## 1. Install dependencies

In [ ]:
!pip install transformers accelerate
!pip install datasets[audio]
!pip install evaluate
!pip install jiwer

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths & config

In [ ]:
import os, sys

DRIVE_ROOT      = '/content/drive/MyDrive'
BASE_DIR        = f'{DRIVE_ROOT}/capstone_design'
SRC_DIR         = f'{BASE_DIR}/whisper_src'
BASE_MODEL      = 'openai/whisper-large-v3-turbo'
TUNED_MODEL     = 'minsu0567/Capstone-Design-Whisper-Dialect'
TEST_JSON       = f'{DRIVE_ROOT}/whisper_test/updated_test_data.json'
TEST_AUDIO_DIR  = f'{DRIVE_ROOT}/whisper_test/audio_files'
EVAL_BATCH_SIZE = 8

assert os.path.isfile(f'{SRC_DIR}/whisper_eval.py'), f'Missing: {SRC_DIR}/whisper_eval.py'
assert os.path.isfile(TEST_JSON), f'Missing: {TEST_JSON}'
assert os.path.isdir(TEST_AUDIO_DIR), f'Missing: {TEST_AUDIO_DIR}'

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print('Paths OK.')

## 4. Holdout test set

In [ ]:
from whisper_eval import load_test_dataset

eval_dataset = load_test_dataset(TEST_JSON, audio_dir=TEST_AUDIO_DIR)

print(f'테스트 샘플: {len(eval_dataset)}개')
print('예시:', eval_dataset[0]['sentence'])

## 5. Base model

In [ ]:
from whisper_eval import evaluate_asr, free_memory, load_whisper

model, processor = load_whisper(BASE_MODEL)

base_scores = evaluate_asr(model, processor, eval_dataset, '베이스 모델',
                           batch_size=EVAL_BATCH_SIZE)

del model, processor
free_memory()

## 6. Fine-tuned model

In [ ]:
model, processor = load_whisper(TUNED_MODEL)

tuned_scores = evaluate_asr(model, processor, eval_dataset, '튜닝 모델',
                            batch_size=EVAL_BATCH_SIZE)

del model, processor
free_memory()

## 7. Comparison

In [ ]:
from whisper_eval import print_comparison, print_samples

print_comparison(base_scores, tuned_scores)
print_samples(eval_dataset, base_scores, tuned_scores, limit=5)